# MLE-STAR × Kaggle：十个基准的 DataOps 实验入口

将 `BENCHMARK_KEY` 改为下方任意一个键，即可下载对应 Kaggle 数据、生成不可变任务 contract、记录数据清单，并运行 MLE-STAR 的安全 `--plan-only` 预检。默认**不会训练或提交**。历史竞赛可能已关闭；只有 Kaggle API 实际接受时才可获得新的 public score。

当前独立 MLE-STAR 分支实现的是搜索、受限代码生成、skrub DataOps DAG、审计与迭代选择。检测、分割和图像去噪的可评分训练 adapter 需要在生成项目通过 OOF/schema 检查后单独提供，不能把此预检 notebook 当作十个竞赛的自动提交器。

In [ ]:
# Clone the isolated MLE-STAR branch. Replace REPO_URL with your fork when needed.
REPO_URL = 'https://github.com/Isso-W/Jiaozi.git'
BRANCH = 'codex/mlestar-kaggle-benchmarks'
!rm -rf /content/Jiaozi
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/Jiaozi
%cd /content/Jiaozi
!python -m pip install -q -r requirements.txt
!python -m mlestar --version

In [ ]:
# Store one KAGGLE_API_TOKEN in Colab Secrets. This cell never prints its value.
import os

try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except ImportError:
    pass  # Running outside Colab: set environment variables before this cell.

assert os.environ.get('KAGGLE_API_TOKEN'), (
    'Add KAGGLE_API_TOKEN to Colab Secrets, then grant this notebook access.'
)
!kaggle --version

In [ ]:
from benchmarks.catalog import BENCHMARKS

for key, benchmark in BENCHMARKS.items():
    print(f'{key:30} {benchmark.competition:45} {benchmark.modality:20} {benchmark.metric.name}')

# Pick exactly one task per run. All ten keys above use the same contract/preflight flow.
BENCHMARK_KEY = 'plant_pathology_2020'
DOWNLOAD_DATA = True  # Set False when the extracted competition files already exist in DATA_ROOT.
DATA_ROOT = '/content/kaggle_data'
RUN_ROOT = '/content/mlestar_runs'
LLM_PROVIDER = 'none'  # 'none' is deterministic and safe; choose 'openai' or 'qwen' only after setting its Secret.

In [ ]:
import json
from pathlib import Path
import subprocess
import sys
import zipfile

from benchmarks.catalog import get_benchmark
from mlestar.contracts import COMPONENT_NAMES, Component, TaskContract

benchmark = get_benchmark(BENCHMARK_KEY)
data_dir = Path(DATA_ROOT) / benchmark.competition
data_dir.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_DATA and not any(data_dir.iterdir()):
    # You must have accepted the competition rules in the Kaggle UI first.
    subprocess.run(['kaggle', 'competitions', 'download', '-c', benchmark.competition, '-p', str(data_dir)], check=True)
    for archive in data_dir.glob('*.zip'):
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(data_dir)

targets = benchmark.labels or benchmark.submission.prediction_columns or ('target',)
task = TaskContract(
    task_id=benchmark.key,
    modality=benchmark.modality,
    target_columns=targets,
    id_column=benchmark.submission.id_column,
    metric=benchmark.metric,
    components=tuple(Component(name) for name in COMPONENT_NAMES),
    description=benchmark.query,
    constraints=(
        'Read Kaggle data as input-only.',
        'Use fixed folds and OOF predictions for model selection.',
        f'Submission contract: {json.dumps(benchmark.submission.to_dict(), sort_keys=True)}',
        f'Fold contract: {json.dumps(benchmark.folds.to_dict(), sort_keys=True)}',
    ),
)
run_dir = Path(RUN_ROOT) / benchmark.key
run_dir.mkdir(parents=True, exist_ok=True)
task_path = run_dir / 'task.input.json'
task_path.write_text(json.dumps(task.to_dict(), indent=2, sort_keys=True), encoding='utf-8')
print({'competition': benchmark.competition, 'data_dir': str(data_dir), 'run_dir': str(run_dir), 'metric': benchmark.metric.to_dict()})

In [ ]:
# This writes inventory.json, candidate source, DataOps audit, and final_report.json.
# It deliberately does not execute model training or make a Kaggle submission.
command = [
    sys.executable, '-m', 'mlestar', 'run',
    '--task', str(task_path),
    '--data-root', str(data_dir),
    '--run-dir', str(run_dir),
    '--plan-only',
    '--llm-provider', LLM_PROVIDER,
]
subprocess.run(command, check=True)
report = json.loads((run_dir / 'final_report.json').read_text(encoding='utf-8'))
assert report['status'] == 'planned'
report

## 进入可评分训练前的检查

先阅读 `projects/candidate_1/audit.jsonl` 和 `final_report.json`，并为当前任务提供能写出固定-fold OOF 预测的训练 adapter。之后才移除 `--plan-only`。在调用 `kaggle competitions submit` 前，必须验证 submission 文件的行数、ID、列名（以及分割任务的 RLE 顺序）与 benchmark contract 一致。